# 品質分析装置データ解析

品質分析装置の計測データを対象としたデータ探索・分析。  
PowerBI ダッシュボードのバックエンドとなるデータパイプラインの検証を兼ねる。

| データセット | 件数 | 主な内容 |
| --- | --- | --- |
| 分析装置出力_詳細.csv | 1,100 | 品質グレード比率 × 100 指標 |
| 品質分析装置データ.csv | 1,046 | 計測値A・計測値B・計測値C |
| 顧客マスタ.csv | 50 | 顧客・規模・地域 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

fm.fontManager.addfont(r'C:\Windows\Fonts\YuGothR.ttc')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['font.family'] = 'Yu Gothic'
plt.rcParams['axes.unicode_minus'] = False

df_detail = pd.read_csv('分析装置出力_詳細.csv', encoding='utf-8-sig')
df_meas   = pd.read_csv('品質分析装置データ.csv', encoding='utf-8-sig')
df_sample = pd.read_csv('サンプル情報.csv', encoding='utf-8-sig')
df_cust   = pd.read_csv('顧客マスタ.csv', encoding='utf-8-sig')

df_meas = df_meas.dropna(subset=['計測値A(%)'])

print(f'分析装置出力: {len(df_detail):,} 件, {len(df_detail.columns)} 指標')
print(f'計測データ  : {len(df_meas):,} 件')
print(f'顧客マスタ  : {len(df_cust):,} 件')

---
## 1. 品質グレード構成比

分析装置が出力する 7 グレードの重量比。主要品質グレード（Grade A）が全体品質の主指標。

In [ ]:
QUALITY_COLS   = ['p_grA_w_ratio','p_grB_w_ratio','p_grC_w_ratio','p_grD_w_ratio',
                  'p_grE_w_ratio','p_grF_w_ratio','p_grG_w_ratio']
QUALITY_LABELS = ['Grade A','Grade B','Grade C','Grade D','Grade E','Grade F','Grade G']
COLORS = ['#1565C0','#E53935','#6D4C41','#FB8C00','#8E24AA','#43A047','#9E9E9E']

means = df_detail[QUALITY_COLS].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(QUALITY_LABELS, means.values, color=COLORS, edgecolor='white', linewidth=0.8)
axes[0].set_title('品質グレード構成比（全サンプル平均）', fontsize=13, pad=10)
axes[0].set_ylabel('重量比 (%)')
axes[0].set_ylim(0, means.max() * 1.18)
for bar, val in zip(bars, means.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

data_primary = df_detail['p_grA_w_ratio'].dropna()
axes[1].hist(data_primary, bins=35, color='#1565C0', edgecolor='white', alpha=0.85)
axes[1].axvline(data_primary.mean(), color='#E53935', linestyle='--', linewidth=1.5,
                label=f'平均 {data_primary.mean():.1f}%')
axes[1].axvline(data_primary.median(), color='#FB8C00', linestyle=':', linewidth=1.5,
                label=f'中央値 {data_primary.median():.1f}%')
axes[1].set_title(f'主要品質グレード分布  (N={len(data_primary):,})', fontsize=13, pad=10)
axes[1].set_xlabel('Grade A 重量比 (%)')
axes[1].set_ylabel('サンプル数')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('docs/images/quality_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grade A  mean={:.1f}%  std={:.1f}%  min={:.1f}%  max={:.1f}%'.format(
    data_primary.mean(), data_primary.std(), data_primary.min(), data_primary.max()))

---
## 2. 計測値A 分析

管理目標値は計測値A **14.5%**。分布と計測値B との相関を確認する。

In [ ]:
TARGET_A = 14.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_meas['計測値A(%)'], bins=30, color='#43A047', edgecolor='white', alpha=0.85)
axes[0].axvline(TARGET_A, color='#E53935', linestyle='--', linewidth=1.8,
                label=f'目標値 {TARGET_A}%')
axes[0].axvline(df_meas['計測値A(%)'].mean(), color='#FB8C00', linestyle=':', linewidth=1.5,
                label=f'平均 {df_meas["計測値A(%)"].mean():.2f}%')
axes[0].set_title(f'計測値A 分布  (N={len(df_meas):,})', fontsize=13, pad=10)
axes[0].set_xlabel('計測値A (%)')
axes[0].set_ylabel('サンプル数')
axes[0].legend(fontsize=10)
over = (df_meas['計測値A(%)'] > TARGET_A).mean() * 100
axes[0].text(0.97, 0.97, f'目標超過: {over:.1f}%',
             transform=axes[0].transAxes, ha='right', va='top', fontsize=10, color='#E53935',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

mask = df_meas['計測値B'].notna() & df_meas['計測値A(%)'].notna()
x = df_meas.loc[mask, '計測値A(%)'].values
y = df_meas.loc[mask, '計測値B'].values
r = np.corrcoef(x, y)[0, 1]
axes[1].scatter(x, y, alpha=0.25, s=8, color='#3949AB')
xs = np.linspace(x.min(), x.max(), 100)
axes[1].plot(xs, np.polyval(np.polyfit(x, y, 1), xs), color='#E53935', linewidth=1.8,
             label=f'回帰直線  r = {r:.3f}')
axes[1].set_title('計測値A vs 計測値B', fontsize=13, pad=10)
axes[1].set_xlabel('計測値A (%)')
axes[1].set_ylabel('計測値B')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('docs/images/measurement_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'計測値A vs 計測値B  Pearson r = {r:.3f}')
print(f'計測値A  mean={df_meas["計測値A(%)"].mean():.2f}%  std={df_meas["計測値A(%)"].std():.2f}%')

---
## 3. 品質指標 相関分析

6 グレード間の相関マトリクスを出力する。

In [ ]:
CORR_COLS   = ['p_grA_w_ratio','p_grB_w_ratio','p_grC_w_ratio',
               'p_grD_w_ratio','p_grE_w_ratio','p_grF_w_ratio']
CORR_LABELS = ['Grade A','Grade B','Grade C','Grade D','Grade E','Grade F']

corr = df_detail[CORR_COLS].corr()
corr.columns = CORR_LABELS
corr.index   = CORR_LABELS

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr, ax=ax,
    annot=True, fmt='.2f', annot_kws={'size': 10},
    cmap='RdBu_r', vmin=-1, vmax=1,
    square=True, linewidths=0.5, linecolor='white',
    cbar_kws={'shrink': 0.8}
)
ax.set_title('品質グレード 相関マトリクス（重量比）', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('docs/images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

top_neg = corr['Grade A'].drop('Grade A').nsmallest(2)
print('Grade A と負の相関 Top2:')
for name, val in top_neg.items():
    print(f'  {name}: r = {val:.3f}')

---
## 4. 顧客規模分布

顧客マスタの規模指標分布。顧客規模感の把握。

In [ ]:
df_scale = df_cust['scale_metric'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_scale, bins=20, color='#00897B', edgecolor='white', alpha=0.85)
axes[0].axvline(df_scale.median(), color='#E53935', linestyle='--', linewidth=1.5,
                label=f'中央値 {df_scale.median():.1f}')
axes[0].set_title(f'顧客規模分布  (N={len(df_scale):,})', fontsize=13, pad=10)
axes[0].set_xlabel('規模指標')
axes[0].set_ylabel('顧客数')
axes[0].legend(fontsize=10)

region_counts = df_cust['region'].value_counts().head(10)
axes[1].barh(region_counts.index[::-1], region_counts.values[::-1],
             color='#1565C0', edgecolor='white', alpha=0.85)
axes[1].set_title('地域別顧客数 (Top 10)', fontsize=13, pad=10)
axes[1].set_xlabel('顧客数')
for i, (idx, val) in enumerate(zip(region_counts.index[::-1], region_counts.values[::-1])):
    axes[1].text(val + 0.1, i, str(val), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('docs/images/customer_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'規模指標  中央値={df_scale.median():.1f}  平均={df_scale.mean():.1f}')
print(f'最大規模: {df_scale.max():.1f}')

---
## まとめ

- Grade A 平均 58.1%  標準偏差 14.3%
- 計測値A 平均 14.34%  標準偏差 0.71%（目標値 14.5%）
- 計測値A vs 計測値B: r = 0.044
- Grade A vs Grade F（最大負相関）: r = −0.06

このノートブックで可視化したデータを PowerBI でダッシュボード化し、  
DAX・PowerQuery を用いてリアルタイム集計・フィルタリングを実装した。